# Business Idea Evaluator

Human-in-the-loop + parallelized multi-agent system built with LangGraph.

> Set your `OPENAI_API_KEY` in a `.env` file before running.

## 1. Imports & setup

In [ ]:
import os, operator
from typing import List, Annotated, Dict
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from IPython.display import Image, display, Markdown

load_dotenv()
llm = ChatOpenAI(model=os.getenv("MODEL_NAME", "gpt-4o"), temperature=0)

## 2. State

`add_messages` appends to the conversation; `operator.or_` merges advisor dicts produced in parallel.

In [ ]:
class State(TypedDict):
    idea: str
    messages: Annotated[List[BaseMessage], add_messages]
    advisor_reports: Annotated[Dict[str, str], operator.or_]
    final_report: str

## 3. Human-in-the-loop nodes

In [ ]:
base_system_msg = SystemMessage(content="""
You are a helpful assistant.
Your job: decide whether you have enough information about the start-up idea.
If not, ask ONE precise follow-up question.
If yes, respond with exactly: DONE
""")

def decide_node(state: State):
    conversation = [base_system_msg] + state["messages"]
    ai_reply: AIMessage = llm.invoke(conversation)   # fixed: .invoke()
    return {"messages": [ai_reply]}

def route(state: State):
    last = state["messages"][-1]
    return "fanout" if last.content.strip().upper().startswith("DONE") else "ask_user_node"

def ask_user_node(state: State):
    question = state["messages"][-1].content
    print(f"\nAssistant: {question}\n")
    human = input("You: ")
    # fixed: return only the new message (add_messages appends it)
    return {"messages": [HumanMessage(content=human)]}

## 4. Advisor nodes (run in parallel)

In [ ]:
def _advisor(role, instructions, state):
    prompt = f"""{instructions}

Idea / conversation so far:
{state['messages']}"""
    report = llm.invoke([SystemMessage(content=prompt)])
    return {"advisor_reports": {role: report.content}}

def market_analyst_advisor(state):
    return _advisor("Market Analyst",
        "You are a senior MARKET ANALYST. Evaluate market potential, competition, "
        "target demographics, and trends. Do market sizing, competitor research, "
        "identify segments, and assess timing/macro trends.", state)

def legal_advisor(state):
    return _advisor("Legal Advisor",
        "You are a LEGAL ADVISOR. Identify IP/licensing/trademark needs, spot "
        "compliance issues (e.g. GDPR), and evaluate contract/partnership considerations.", state)

def technical_advisor(state):
    return _advisor("Technical Advisor",
        "You are a TECHNICAL/PRODUCT FEASIBILITY EXPERT. Estimate development "
        "complexity/time, recommend tech stacks, and evaluate infra/scalability/cost risk.", state)

def strategist_advisor(state):
    return _advisor("Strategist Advisor",
        "You are a STRATEGIST ADVISOR. Define launch milestones, select distribution "
        "channels and positioning, and craft early traction tactics.", state)

## 5. Collect & report

In [ ]:
def collect_and_report(state: State):
    if len(state["advisor_reports"]) < 4:
        return {}
    prompt = f"""You are a senior consultant. Combine the advisor notes below into
one clear, structured evaluation report for the founder.

{state['advisor_reports']}"""
    report = llm.invoke(prompt).content
    display(Markdown("### FINAL REPORT\n\n" + report))
    return {"final_report": report}

## 6. Build the graph

In [ ]:
builder = StateGraph(State)
builder.add_node("decide_node", decide_node)
builder.add_node("ask_user_node", ask_user_node)
builder.add_node("fanout", lambda state: {})
builder.add_node("market_analyst_advisor", market_analyst_advisor)
builder.add_node("legal_advisor", legal_advisor)
builder.add_node("technical_advisor", technical_advisor)
builder.add_node("strategist_advisor", strategist_advisor)
builder.add_node("collect_and_report", collect_and_report)

builder.set_entry_point("decide_node")
builder.add_edge("ask_user_node", "decide_node")
builder.add_conditional_edges("decide_node", route,
    {"ask_user_node": "ask_user_node", "fanout": "fanout"})
for a in ["market_analyst_advisor","legal_advisor","technical_advisor","strategist_advisor"]:
    builder.add_edge("fanout", a)
    builder.add_edge(a, "collect_and_report")
builder.add_edge("collect_and_report", END)

graph = builder.compile()
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

## 7. Run it

In [ ]:
print("What is your business idea?")
idea = input("You: ")
init_state: State = {"idea": idea, "messages": [HumanMessage(content=idea)],
                     "advisor_reports": {}, "final_report": ""}
graph.invoke(init_state, config={"configurable": {"thread_id": "run-1"}});